Phase 1 : Charger et explorer

Chargez un dataset de classification fourni par scikit-learn ( load_breast_cancer est un bon choix : 569
patients, 30 mesures, tumeur bénigne ou maligne). Inspectez-le avant toute chose.

In [1]:
from sklearn.datasets import load_breast_cancer
import numpy as np

In [11]:
def explorer_dataset():
    """Charge le dataset et affiche ses caractéristiques de base.
    Doit afficher : nombre de lignes, nombre de colonnes,
    les classes possibles et leur répartition (équilibrée ou non ?).
    """
    # TODO : charger le dataset (return_X_y=True donne X et y séparés)
    X, y = load_breast_cancer(return_X_y=True)
    # TODO : afficher la forme de X (combien de lignes, combien de colonnes)
    print(f"Lignes, colonnes : {X.shape}")
    # TODO : compter et afficher combien d'exemples dans chaque classe
    cas_malignes = (y == 0).sum()
    cas_benignes = (y == 1).sum()
    print(f"Classe 0 (maligne) : {cas_malignes} cas")
    print(f"Classe 1 (bénigne) : {cas_benignes} cas")
    pass

In [12]:
explorer_dataset()

Lignes, colonnes : (569, 30)
Classe 0 (maligne) : 212 cas
Classe 1 (bénigne) : 357 cas


Phase 2 : Le pipeline supervisé complet

Reprenez les 5 étapes du matin sur ce dataset : split, entraînement, prédiction, mesure. Encapsulez dans une
fonction qui prend un modèle en argument.

In [21]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

In [23]:
def entrainer_et_evaluer(modele, X_train, X_test, y_train, y_test):
    """Entraîne le modèle, prédit sur le test, renvoie l'accuracy.
    Doit renvoyer un float entre 0 et 1.
    """
    # TODO : entraîner le modèle sur les données d'apprentissage
    modele.fit(X_train, y_train)
    
    # TODO : prédire sur le jeu de test
    predictions = modele.predict(X_test)
    
    # TODO : calculer l'accuracy et la renvoyer
    score = accuracy_score(y_test, predictions)
    
    return score
    pass

In [25]:
# Charger les données 
X, y = load_breast_cancer(return_X_y=True)

# Séparer les données
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# On crée notre modèle
mon_modele = DecisionTreeClassifier(random_state=42)

# On utilise ta nouvelle fonction
resultat = entrainer_et_evaluer(mon_modele, X_train, X_test, y_train, y_test)

print(f"Le score de précision (accuracy) est de : {resultat}")

Le score de précision (accuracy) est de : 0.9473684210526315


Phase 3 : L'Arène (premier classement)

Faites s'affronter au moins 3 algos sur le MÊME découpage train/test. Affichez un classement trié par accuracy.

In [26]:
from sklearn.neighbors import KNeighborsClassifier

In [38]:
def arene(X_train, X_test, y_train, y_test):
    """Entraîne plusieurs modèles, renvoie un classement trié.
    Doit afficher un tableau lisible : nom de l'algo, accuracy.
    """
    # TODO : construire un dictionnaire {nom: modèle} avec au moins 3 algos
    # (repensez à ceux du matin : arbre, régression logistique, KNN...)
    modeles = {
        "Arbre de décision": DecisionTreeClassifier(random_state=42),
        "KNN": KNeighborsClassifier(),
        "Régression logistique": LogisticRegression(max_iter=10000)
    }
    # TODO : pour chaque modèle, réutiliser entrainer_et_evaluer sur le MÊME split
    results = []
    for nom, modele in modeles.items():
        accuracy = entrainer_et_evaluer(modele, X_train, X_test, y_train, y_test)
        results.append((nom, accuracy))

    results.sort(reverse=True)

    # TODO : trier par accuracy décroissante et afficher le podium
    print("Classement des modèles :")
    for nom, accuracy in results:
        resultat_percent = accuracy * 100
        print(f"{nom} : {resultat_percent:.1f}%")

    pass


In [39]:
arene(X_train, X_test, y_train, y_test)

Classement des modèles :
Régression logistique : 95.6%
KNN : 95.6%
Arbre de décision : 94.7%


Si on ne met pas `max_iter=10000`, la Régression Logistique affiche une erreur

**Pourquoi ?**
La régression logistique n'a pas réussi à converger en 100 étapes, car les variables ont des échelles très différentes, ce qui ralentit son apprentissage.

**Que se passe-t-il si on augmente max_iter ?**
En augmentant max_iter à 10000, on donne plus de temps à l'algorithme pour trouver la solution.

Phase 4 : Bascule non-supervisé

Maintenant, cachez les étiquettes. Lancez un KMeans qui doit trouver 2 groupes tout seul, sans jamais voir y .
Puis comparez : les groupes trouvés correspondent-ils aux vraies classes (bénigne/maligne) ?

In [35]:
from sklearn.cluster import KMeans

In [40]:
def clustering_aveugle(X):
    """Regroupe les données en 2 clusters sans les étiquettes.
    Renvoie les labels de cluster attribués à chaque point.
    """
    # TODO : créer un KMeans à 2 clusters et renvoyer le cluster de chaque point
    # (indice : une seule méthode fait l'entraînement ET la prédiction d'un coup)
    kmeans = KMeans(n_clusters=2, random_state=42)
    clusters = kmeans.fit_predict(X)
    return clusters
    pass

In [ ]:
# On lance notre fonction sur tout le dataset X
labels_kmeans = clustering_aveugle(X)

# On calcule le nombre total de patients
total_patients = len(y)

# KMeans a utilisé la même logique que nous (0 = maligne, 1 = bénigne)
match_direct = (labels_kmeans == y).sum()

# On garde le meilleur des deux scores
pourcentage_reussite = (match_direct / total_patients) * 100

print(f"L'algorithme aveugle a retrouvé les bonnes classes à {pourcentage_reussite:.1f}% !")

L'algorithme aveugle a retrouvé les bonnes classes à 85.4% !


Le clustering a bien réussi (≈85%). Ce n’est pas de la chance, les données contiennent une vraie structure. Les tumeurs bénignes et malignes ont des caractéristiques différentes, et KMeans a réussi à les regrouper correctement.